# 06 Candidate Model Training

Train lightweight YOLO candidate architectures on the same full-pipeline dataset for architecture triage.


## Purpose

This notebook compares candidate YOLO architectures using `data_exp_D_full_pipeline.yaml` only. Every candidate sees the same train/validation split and the same online augmentation settings.

This is architecture triage, not the augmentation ablation study. The untouched test set is not used to select the candidate architecture.


## Setup

Find the `v2_pipeline` root, add it to `sys.path`, and import training/evaluation helpers from `src/training`.


In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import display


def find_v2_root(start: Path) -> Path:
    """Find the v2_pipeline root from the current working directory."""
    for candidate in (start, *start.parents):
        if candidate.name == "v2_pipeline" and (candidate / "src").exists():
            return candidate
        nested = candidate / "ppe-detection" / "v2_pipeline"
        if nested.exists() and (nested / "src").exists():
            return nested
    raise RuntimeError("Could not find v2_pipeline root")


V2_ROOT = find_v2_root(Path.cwd().resolve())
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from src.training.evaluate_yolo import (
    benchmark_model_latency,
    evaluate_candidate_model,
    rank_candidate_results,
)
from src.training.train_yolo import train_candidate_models

V2_ROOT

WindowsPath('C:/Github/smart-factory-safety-monitoring-system/ppe-detection/v2_pipeline')

The notebook imports orchestration helpers only. Ultralytics itself is imported inside the helper functions when training, validation, or benchmarking actually runs, so config and JSON checks can run without starting model work.

The OpenMP environment variables are set before training libraries load. This avoids a Windows kernel crash caused by duplicate OpenMP runtimes such as `libomp.dll` and `libiomp5md.dll` being initialized in the same process.


## Load Configuration

Load training settings, online augmentation settings, and class metadata.


In [2]:
training_config_path = V2_ROOT / "configs" / "training_config.yaml"
augmentation_config_path = V2_ROOT / "configs" / "augmentation_config.yaml"
class_names_path = V2_ROOT / "configs" / "class_names.yaml"

with training_config_path.open("r", encoding="utf-8") as file_handle:
    training_config = yaml.safe_load(file_handle)
with augmentation_config_path.open("r", encoding="utf-8") as file_handle:
    augmentation_config = yaml.safe_load(file_handle)
with class_names_path.open("r", encoding="utf-8") as file_handle:
    class_config = yaml.safe_load(file_handle)

data_yaml = V2_ROOT / "data_exp_D_full_pipeline.yaml"
candidate_models = list(training_config["candidate_models"])
TRIAGE_EPOCHS = int(training_config.get("candidate_epochs", training_config["epochs"]))

config_summary = pd.DataFrame(
    [
        {"setting": "data_yaml", "value": str(data_yaml)},
        {"setting": "candidate_models", "value": candidate_models},
        {"setting": "triage_epochs", "value": TRIAGE_EPOCHS},
        {
            "setting": "selection_priority",
            "value": training_config.get("selection_priority", []),
        },
        {"setting": "classes", "value": class_config["names"]},
    ]
)
display(config_summary)

,setting,value
0,data_yaml,C:\Github\smart-factory-safety-monitoring-syst...
1,candidate_models,"[yolov8n.pt, yolov9t.pt, yolov10n.pt, yolo11n...."
2,triage_epochs,50
3,selection_priority,"[recall, map50, map50_95, fps, model_size]"
4,classes,"{0: 'person', 1: 'helmet', 2: 'vest'}"


`candidate_epochs` is used for architecture triage when present. The original `epochs` value remains available for final training later, so Notebook 06 can be shorter without changing the meaning of the final training config.


## Dataset Safety Check

Confirm the full-pipeline experiment YAML and referenced folders exist before training starts.


In [3]:
if not data_yaml.exists():
    raise FileNotFoundError(f"Dataset YAML not found: {data_yaml}")

with data_yaml.open("r", encoding="utf-8") as file_handle:
    data_config = yaml.safe_load(file_handle)

dataset_root = V2_ROOT / data_config["path"]
dataset_checks = pd.DataFrame(
    [
        {
            "split": split,
            "path": str(dataset_root / data_config[split]),
            "exists": (dataset_root / data_config[split]).exists(),
        }
        for split in ["train", "val", "test"]
    ]
)
display(dataset_checks)

missing_paths = dataset_checks.loc[~dataset_checks["exists"], "path"].tolist()
if missing_paths:
    raise FileNotFoundError(f"Experiment dataset folders are missing: {missing_paths}")

,split,path,exists
0,train,C:\Github\smart-factory-safety-monitoring-syst...,True
1,val,C:\Github\smart-factory-safety-monitoring-syst...,True
2,test,C:\Github\smart-factory-safety-monitoring-syst...,True


The test path is checked only for dataset completeness. It is not used for ranking candidates in this notebook; candidate selection comes from validation metrics.


## Candidate Models

Display the candidate checkpoints loaded from `configs/training_config.yaml`.


In [4]:
candidate_table = pd.DataFrame(
    [
        {"order": index + 1, "model_name": model_name}
        for index, model_name in enumerate(candidate_models)
    ]
)
display(candidate_table)

,order,model_name
0,1,yolov8n.pt
1,2,yolov9t.pt
2,3,yolov10n.pt
3,4,yolo11n.pt
4,5,yolo12n.pt
5,6,yolo26n.pt


If a checkpoint is unavailable or unsupported by the installed Ultralytics version, the training helper records that model as failed and continues to the next candidate.


## Training Arguments

Build shared Ultralytics training arguments for every candidate model.


In [5]:
online_aug = augmentation_config["online_augmentation"]

# Set DRY_RUN=True only when checking notebook wiring without training models.
DRY_RUN = False
OVERWRITE_EXISTING_RUNS = False

train_args = {
    "epochs": TRIAGE_EPOCHS,
    "imgsz": int(training_config["imgsz"]),
    "batch": int(training_config["batch"]),
    "device": training_config["device"],
    "workers": int(training_config["workers"]),
    "patience": int(training_config["patience"]),
    "seed": int(training_config["seed"]),
    "amp": bool(training_config.get("amp", False)),
    "hsv_h": float(online_aug["hsv_h"]),
    "hsv_s": float(online_aug["hsv_s"]),
    "hsv_v": float(online_aug["hsv_v"]),
    "degrees": float(online_aug["degrees"]),
    "translate": float(online_aug["translate"]),
    "scale": float(online_aug["scale"]),
    "perspective": float(online_aug["perspective"]),
    "fliplr": float(online_aug["fliplr"]),
    "flipud": float(online_aug["flipud"]),
    "mosaic": float(online_aug["mosaic"]),
    "mixup": float(online_aug["mixup"]),
    "close_mosaic": int(online_aug["close_mosaic"]),
    "plots": True,
    "dry_run": DRY_RUN,
    "overwrite": OVERWRITE_EXISTING_RUNS,
}

display(
    pd.DataFrame(
        [{"argument": key, "value": value} for key, value in train_args.items()]
    )
)

,argument,value
0,epochs,50
1,imgsz,640
2,batch,24
3,device,0
4,workers,8
5,patience,20
6,seed,42
7,amp,True
8,hsv_h,0.015
9,hsv_s,0.3


Online augmentation is passed as training arguments, not encoded into the dataset YAML. This keeps the experiment dataset definition clean and lets later ablation notebooks turn online augmentation on or off explicitly.

`workers=0` is the safest Windows/Jupyter setting because PyTorch dataloader workers can survive a kernel crash as orphan `--multiprocessing-fork` Python processes. After a successful small run, you can raise `workers` in `training_config.yaml` from a terminal script if you want more throughput.


## Train Candidate Models

Train every candidate on `exp_D_full_pipeline` and collect training-run validation metrics.


In [6]:
candidate_runs_dir = V2_ROOT / "runs" / "candidate_models"

candidate_results = train_candidate_models(
    data_yaml=data_yaml,
    model_names=candidate_models,
    output_dir=candidate_runs_dir,
    train_args=train_args,
    continue_on_error=True,
)

display(candidate_results)

New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.51  Python-3.10.20 torch-2.13.0.dev20260517+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\runs\candidate_models\_data_exp_D_full_pipeline_resolved.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.3, hsv_v=0.35, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.

,model_name,run_name,status,run_dir,precision,recall,map50,map50_95,fitness,training_time_seconds,best_weights_path,last_weights_path,model_size_mb,error_message,notes
0,yolov8n.pt,YOLOv8_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.89514,0.82358,0.86476,0.56785,None,275.848,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.953348,,
1,yolov9t.pt,YOLOv9_Tiny_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.88885,0.85789,0.87907,0.55896,None,413.505,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.416207,,
2,yolov10n.pt,YOLOv10_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.87827,0.80336,0.85335,0.55490,None,364.165,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.479425,,
3,yolo11n.pt,YOLO11_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.91118,0.81360,0.87197,0.57598,None,287.523,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.211817,,
4,yolo12n.pt,YOLO12_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.91032,0.81483,0.87648,0.56962,None,388.287,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.256006,,
5,yolo26n.pt,YOLO26_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.87171,0.79268,0.85315,0.54279,None,365.658,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.131291,,


Runs are saved under `runs/candidate_models/`. When `OVERWRITE_EXISTING_RUNS=False`, reruns use a clear numeric suffix instead of overwriting an existing run folder.


## Optional Validation Refresh and Latency Benchmark

Optionally refresh validation metrics from best weights and estimate inference latency on validation images.


In [7]:
RUN_POST_TRAIN_VALIDATION = False
RUN_LATENCY_BENCHMARK = True
MAX_BENCHMARK_IMAGES = 20

val_images_dir = dataset_root / data_config["val"]

for row_index, row in candidate_results.iterrows():
    if row["status"] != "trained":
        continue
    best_weights = Path(str(row.get("best_weights_path", "")))
    if RUN_POST_TRAIN_VALIDATION:
        eval_metrics = evaluate_candidate_model(
            weights_path=best_weights,
            data_yaml=data_yaml,
            imgsz=int(training_config["imgsz"]),
            device=training_config["device"],
        )
        for metric_name in ["precision", "recall", "map50", "map50_95", "fitness"]:
            if eval_metrics.get(metric_name) is not None:
                candidate_results.at[row_index, metric_name] = eval_metrics[metric_name]
        candidate_results.at[row_index, "eval_status"] = eval_metrics["status"]
        candidate_results.at[row_index, "eval_error_message"] = eval_metrics[
            "error_message"
        ]

    if RUN_LATENCY_BENCHMARK:
        latency_metrics = benchmark_model_latency(
            weights_path=best_weights,
            sample_images_dir=val_images_dir,
            imgsz=int(training_config["imgsz"]),
            device=training_config["device"],
            max_images=MAX_BENCHMARK_IMAGES,
        )
        for metric_name, metric_value in latency_metrics.items():
            candidate_results.at[row_index, metric_name] = metric_value

display(candidate_results)

,model_name,run_name,status,run_dir,precision,recall,map50,map50_95,fitness,training_time_seconds,best_weights_path,last_weights_path,model_size_mb,error_message,notes,latency_status,benchmark_images,avg_inference_ms,fps,latency_error
0,yolov8n.pt,YOLOv8_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.89514,0.82358,0.86476,0.56785,None,275.848,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.953348,,,benchmarked,20.0,50.243315,19.903145,
1,yolov9t.pt,YOLOv9_Tiny_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.88885,0.85789,0.87907,0.55896,None,413.505,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,4.416207,,,benchmarked,20.0,55.675875,17.961101,
2,yolov10n.pt,YOLOv10_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.87827,0.80336,0.85335,0.55490,None,364.165,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.479425,,,benchmarked,20.0,48.120415,20.781201,
3,yolo11n.pt,YOLO11_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.91118,0.81360,0.87197,0.57598,None,287.523,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.211817,,,benchmarked,20.0,49.378115,20.251887,
4,yolo12n.pt,YOLO12_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.91032,0.81483,0.87648,0.56962,None,388.287,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.256006,,,benchmarked,20.0,51.323045,19.484425,
5,yolo26n.pt,YOLO26_Nano_expD,trained,C:\Github\smart-factory-safety-monitoring-syst...,0.87171,0.79268,0.85315,0.54279,None,365.658,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,5.131291,,,benchmarked,20.0,49.056750,20.384555,


Latency benchmarking uses validation images only and is treated as an engineering signal after the validation metrics. If benchmarking fails on a machine, the error is recorded without changing the candidate training results.


## Save Candidate Reports

Save results, failures, and ranked candidates under `reports/training`.


In [8]:
reports_dir = V2_ROOT / "reports" / "training"
reports_dir.mkdir(parents=True, exist_ok=True)

ranked_results = rank_candidate_results(candidate_results)
failures_df = candidate_results.loc[candidate_results["status"].eq("failed")].copy()

candidate_results.to_csv(reports_dir / "candidate_model_results.csv", index=False)
failures_df.to_csv(reports_dir / "candidate_model_failures.csv", index=False)
ranked_results.to_csv(reports_dir / "candidate_model_ranking.csv", index=False)

print(f"Saved candidate reports to: {reports_dir}")

Saved candidate reports to: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\training


The failures report may be empty. That is fine; it exists so unsupported checkpoint names or unavailable weights are still documented when they happen.


## Ranked Candidate Table

Rank candidates by recall, mAP50, mAP50-95, FPS, then model size.


In [9]:
display_columns = [
    "model_name",
    "run_name",
    "status",
    "recall",
    "map50",
    "map50_95",
    "fps",
    "avg_inference_ms",
    "model_size_mb",
    "rank_score",
    "error_message",
]
existing_columns = [
    column for column in display_columns if column in ranked_results.columns
]
display(ranked_results[existing_columns])

,model_name,run_name,status,recall,map50,map50_95,fps,avg_inference_ms,model_size_mb,rank_score,error_message
0,yolov9t.pt,YOLOv9_Tiny_expD,trained,0.85789,0.87907,0.55896,17.961101,55.675875,4.416207,8.587696e+08,
1,yolov8n.pt,YOLOv8_Nano_expD,trained,0.82358,0.86476,0.56785,19.903145,50.243315,5.953348,8.244453e+08,
2,yolo12n.pt,YOLO12_Nano_expD,trained,0.81483,0.87648,0.56962,19.484425,51.323045,5.256006,8.157071e+08,
3,yolo11n.pt,YOLO11_Nano_expD,trained,0.81360,0.87197,0.57598,20.251887,49.378115,5.211817,8.144726e+08,
4,yolov10n.pt,YOLOv10_Nano_expD,trained,0.80336,0.85335,0.55490,20.781201,48.120415,5.479425,8.042139e+08,
5,yolo26n.pt,YOLO26_Nano_expD,trained,0.79268,0.85315,0.54279,20.384555,49.056750,5.131291,7.935337e+08,


The ranking table intentionally emphasizes recall first because missed PPE/person detections are the highest-risk failure mode for the safety pipeline.


## Optional Figures

Save compact comparison charts for reports when metrics are available.


In [10]:
figures_dir = V2_ROOT / "reports" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)


def save_metric_barplot(
    df: pd.DataFrame, metric: str, output_name: str, title: str
) -> None:
    plot_df = (
        df.loc[df[metric].notna()].copy() if metric in df.columns else pd.DataFrame()
    )
    if plot_df.empty:
        print(f"Skipping {output_name}: no {metric} values available.")
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(plot_df["model_name"], plot_df[metric])
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    fig.savefig(figures_dir / output_name, dpi=140)
    plt.close(fig)
    print(f"Saved {figures_dir / output_name}")


save_metric_barplot(
    ranked_results, "map50", "candidate_model_map50.png", "Candidate mAP50"
)
save_metric_barplot(
    ranked_results, "recall", "candidate_model_recall.png", "Candidate recall"
)
save_metric_barplot(
    ranked_results,
    "avg_inference_ms",
    "candidate_model_latency.png",
    "Candidate latency",
)

Saved C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\figures\candidate_model_map50.png
Saved C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\figures\candidate_model_recall.png
Saved C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\figures\candidate_model_latency.png


These figures are generated report artifacts. They are useful for slides and written reports, but the CSV ranking remains the source of truth.


## Checklist Before Notebook 07

- All candidates were trained or safely logged as failed.
- Every candidate used `data_exp_D_full_pipeline.yaml`.
- Candidate ranking used validation metrics only.
- Test metrics were not used for architecture selection.
- Candidate reports were saved under `reports/training`.
- The best architecture is selected before starting the ablation study.
